# Fine-tune VLA-Adapter on RobotArmLearning

This notebook downloads the public RobotArmLearning demonstrations from Hugging Face, regenerates the shoulder and wrist observations in headless MuJoCo, converts them to RLDS, and LoRA-fine-tunes the Qwen2.5-0.5B VLA-Adapter model.

Before running, select **Runtime → Change runtime type** and pick an **L4 or A100**. VLA-Adapter trains in bfloat16 throughout, which needs an Ampere-or-newer GPU; the free-tier T4 is Turing and PyTorch will refuse the first autocast. The preflight cell checks this. Allow roughly 30 GB of runtime disk. Preview `sim.mp4` files are intentionally not downloaded; the compact state trajectories are sufficient to regenerate the training images.

In [ ]:
# User settings
DATASET_REPO = "FoxNerdSaysMoo/robot-arm-learning-data"
DATASET_REVISION = "main"  # Replace with a commit SHA to freeze the dataset.
CODE_REPO = "https://github.com/zebulontaylor/RobotArmTraining.git"
VLA_REPO = "https://github.com/OpenHelix-Team/VLA-Adapter.git"
VLA_COMMIT = "23fa0c9c159e2aa04341cdd3e924f44061311060"
MODEL_REPO = "Stanford-ILIAD/prism-qwen25-extra-dinosiglip-224px-0_5b"
INSTRUCTION = "stack the three colored cubes"
SAMPLE_HZ = 10.0
MAX_STEPS = 10_000       # Use 50 first for a quick end-to-end test.
SAVE_FREQ = 1_000
BATCH_SIZE = 2           # Fits a 16 GB GPU with gradient checkpointing.
GRAD_ACCUM_STEPS = 4     # Effective batch size: 2 x 4 = 8.
SAVE_TO_DRIVE = False    # True preserves checkpoints across runtime resets.
DRIVE_OUTPUT = "/content/drive/MyDrive/robot-arm-learning-vla-outputs"


In [ ]:
# GPU and disk preflight
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU detected. Enable a GPU runtime before continuing.")
subprocess.run(["nvidia-smi"], check=True)

# VLA-Adapter casts weights, inputs and autocast to bfloat16 in a dozen places
# and carries no GradScaler, so a pre-Ampere GPU cannot run it unconverted.
try:
    import torch
    capability = torch.cuda.get_device_capability()
except Exception:
    capability = None
if capability is None:
    print("Could not read compute capability; skipping the bfloat16 check.")
elif capability[0] < 8:
    raise RuntimeError(
        f"{torch.cuda.get_device_name(0)} is compute capability {capability[0]}.{capability[1]}, "
        "which has no bfloat16 support. Choose an L4 or A100 runtime.")
else:
    print(f"Compute capability {capability[0]}.{capability[1]}: bfloat16 supported.")

free_gb = shutil.disk_usage("/content").free / 2**30
print(f"Free runtime disk: {free_gb:.1f} GiB")
if free_gb < 30:
    raise RuntimeError("At least 30 GiB of free runtime disk is recommended.")


## 1. Fetch code and create an isolated Python 3.10 environment

VLA-Adapter pins PyTorch 2.2 and TensorFlow 2.15. The separate environment avoids conflicts with Colab's preinstalled packages. Long setup output is normal.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path("/content")
CODE_DIR = ROOT / "RobotArmLearning"
VLA_DIR = CODE_DIR / "VLA-Adapter"
VENV = ROOT / "vla-env"
PYTHON = str(VENV / "bin/python")

def run(command, **kwargs):
    """Run a subprocess, streaming its output into the notebook.

    A child process writes to file descriptor 1, which bypasses the
    `sys.stdout` that ipykernel replaces, so `subprocess.run` output is
    invisible here. Reading the pipe ourselves keeps progress lines -- and
    failures -- in the cell where they can be seen.
    """
    print("+", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               **kwargs)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

if not CODE_DIR.exists():
    run(["git", "clone", "--depth=1", CODE_REPO, CODE_DIR])
if not VLA_DIR.exists():
    run(["git", "clone", VLA_REPO, VLA_DIR])
run(["git", "checkout", VLA_COMMIT], cwd=VLA_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
run(["uv", "python", "install", "3.10"])
if not VENV.exists():
    run(["uv", "venv", VENV, "--python", "3.10"])
# protobuf has to land on 4.x: TensorFlow 2.15 caps it below 5, and wandb
# 0.26+ dropped the generated bindings for protobuf 3. Nothing here pins it,
# and current tensorflow_metadata resolves to a release whose *_pb2.py modules
# import google.protobuf.runtime_version -- protobuf 5.27 and later only.
# 1.13.1 is the newest that both predates that and permits 4.x; the tfmd
# releases in between cap protobuf below 4.21 and so break wandb instead.
run(["uv", "pip", "install", "--python", PYTHON, "-e", VLA_DIR,
     "tensorflow-metadata==1.13.1", "protobuf==4.25.9",
     "mujoco", "opencv-python-headless", "scipy"])
run([PYTHON, "-c", "import torch, tensorflow as tf, mujoco; "
     "print('torch', torch.__version__, 'tensorflow', tf.__version__, 'mujoco', mujoco.__version__)"])


## 2. Register RobotArmLearning with VLA-Adapter

The upstream trainer only knows its built-in Open-X datasets. This idempotent patch registers RobotArmLearning's two cameras, 8-D proprioceptive state, and 7-D end-effector action. It also enables optional gradient checkpointing, prevents duplicate checkpoint writes during gradient accumulation, and makes the FlashAttention 2 request conditional on the GPU actually supporting it.

In [ ]:
def insert_after(path, anchor, addition, marker):
    text = path.read_text()
    if marker not in text:
        if anchor not in text:
            raise RuntimeError(f"Patch anchor not found in {path}")
        path.write_text(text.replace(anchor, anchor + addition, 1))

oxe = VLA_DIR / "prismatic/vla/datasets/rlds/oxe"
insert_after(
    oxe / "configs.py",
    "OXE_DATASET_CONFIGS = {\n",
    '    "robot_arm_learning_panthera": {\n'
    '        "image_obs_keys": {"primary": "image", "secondary": None, "wrist": "wrist_image"},\n'
    '        "depth_obs_keys": {"primary": None, "secondary": None, "wrist": None},\n'
    '        "state_obs_keys": ["state"],\n'
    '        "state_encoding": StateEncoding.POS_EULER,\n'
    '        "action_encoding": ActionEncoding.EEF_POS,\n'
    '    },\n',
    '"robot_arm_learning_panthera"',
)
insert_after(
    oxe / "mixtures.py",
    "OXE_NAMED_MIXTURES: Dict[str, List[Tuple[str, float]]] = {\n",
    '    "robot_arm_learning_panthera": [("robot_arm_learning_panthera", 1.0)],\n',
    '"robot_arm_learning_panthera"',
)
insert_after(
    oxe / "transforms.py",
    "OXE_STANDARDIZATION_TRANSFORMS = {\n",
    '    "robot_arm_learning_panthera": lambda trajectory: trajectory,\n',
    '"robot_arm_learning_panthera"',
)

# The Qwen2.5 backbone asks for FlashAttention 2 unconditionally. VLA-Adapter
# leaves flash_attn to a manual post-install step, and it needs an Ampere or
# newer GPU regardless -- a T4 is Turing and cannot run it at all. Fall back to
# PyTorch SDPA unless both the hardware and the package are present.
base_llm = VLA_DIR / "prismatic/models/backbones/llm/base_llm.py"
insert_after(
    base_llm,
    "        return self.tokenizer.pad_token_id\n\n\n",
    "def flash_attention_2_available() -> bool:\n"
    '    """FlashAttention 2 needs an Ampere-or-newer GPU and the flash_attn package."""\n'
    "    import importlib.util\n"
    "    if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:\n"
    "        return False\n"
    '    return importlib.util.find_spec("flash_attn") is not None\n\n\n',
    "def flash_attention_2_available",
)
attention = base_llm.read_text()
old_attn = "use_flash_attention_2=use_flash_attention_2 if not self.inference_mode else False,"
new_attn = ("use_flash_attention_2=use_flash_attention_2 and flash_attention_2_available()\n"
            "                if not self.inference_mode else False,")
if old_attn in attention:
    base_llm.write_text(attention.replace(old_attn, new_attn, 1))

finetune = VLA_DIR / "vla-scripts/finetune.py"
insert_after(
    finetune,
    "    use_pro_version: bool = True                             # the version number\n",
    "    use_gradient_checkpointing: bool = False\n",
    "use_gradient_checkpointing: bool",
)
checkpoint_anchor = "    # FiLM setup\n"
checkpoint_block = (
    "    if cfg.use_gradient_checkpointing:\n"
    "        llm = (vla.base_model.model if cfg.use_lora else vla).language_model\n"
    "        llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": False})\n"
    "        llm.config.use_cache = False\n"
    "        print(\"Gradient checkpointing enabled on language_model\")\n\n"
)
text = finetune.read_text()
if "Gradient checkpointing enabled on language_model" not in text:
    if checkpoint_anchor not in text:
        raise RuntimeError("Gradient-checkpointing patch anchor not found")
    text = text.replace(checkpoint_anchor, checkpoint_block + checkpoint_anchor, 1)
old_save = "if gradient_step_idx > 0 and log_step % cfg.save_freq == 0:"
new_save = ("if gradient_step_idx > 0 and log_step % cfg.save_freq == 0 "
            "and (batch_idx + 1) % cfg.grad_accumulation_steps == 0:")
if old_save in text:
    text = text.replace(old_save, new_save, 1)
finetune.write_text(text)
print("RobotArmLearning adapter registration is ready.")


## 3. Download and validate demonstrations

Only `data.npz` and `meta.json` are fetched. Re-running this cell resumes from the Hugging Face cache.

In [ ]:
download_program = f'''
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id={DATASET_REPO!r},
    repo_type="dataset",
    revision={DATASET_REVISION!r},
    local_dir={str(CODE_DIR)!r},
    allow_patterns=["data/episode_*/data.npz", "data/episode_*/meta.json"],
)
'''
run([PYTHON, "-c", download_program])

validate_program = f'''
from pathlib import Path
import numpy as np
root = Path({str(CODE_DIR / 'data')!r})
episodes = sorted(root.glob("episode_*"))
required = {{"t", "q", "ctrl", "ee_pos", "ee_quat", "obj_pos", "obj_quat", "gripper"}}
if not episodes:
    raise RuntimeError("No episodes downloaded")
for episode in episodes:
    with np.load(episode / "data.npz") as data:
        missing = required - set(data.files)
        if missing:
            raise RuntimeError(f"{{episode.name}} is missing {{sorted(missing)}}")
print(f"Validated {{len(episodes)}} episodes")
'''
run([PYTHON, "-c", validate_program])


## 4. Render shoulder/wrist observations and build RLDS

The next cell makes sure MuJoCo renders on the GPU. Colab ships the NVIDIA
driver but usually omits the libglvnd vendor file that advertises it to EGL,
and MuJoCo then falls back to Mesa's `llvmpipe` software rasteriser without
warning. The images come out correct, roughly a hundred times slower --
minutes of rendering become hours. The check is a hard failure rather than a
warning for that reason.

Rendering is resumable episode by episode. The generated JPEGs and TFRecords
remain on the ephemeral Colab disk; only checkpoints need to be preserved.

In [ ]:
# Headless GPU rendering, verified rather than assumed.
NVIDIA_ICD = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not NVIDIA_ICD.exists():
    NVIDIA_ICD.parent.mkdir(parents=True, exist_ok=True)
    NVIDIA_ICD.write_text('{"file_format_version": "1.0.0", '
                          '"ICD": {"library_path": "libEGL_nvidia.so.0"}}')
    print("wrote", NVIDIA_ICD)

GL_PROBE = """
import os
os.environ["MUJOCO_GL"] = "egl"
from mujoco.egl import GLContext
from OpenGL import GL
context = GLContext(64, 64)  # Bound to a name: a temporary is freed
context.make_current()       # before glGetString can be answered.
print("renderer", GL.glGetString(GL.GL_RENDERER).decode())
"""
SOFTWARE_GL = ("llvmpipe", "softpipe", "swrast")

def gl_renderer(env):
    """The OpenGL renderer MuJoCo's EGL context lands on, or None if it fails."""
    probe = subprocess.run([PYTHON, "-c", GL_PROBE], env=env,
                           capture_output=True, text=True)
    for line in probe.stdout.splitlines():
        if line.startswith("renderer "):
            return line[len("renderer "):]
    return None

def is_software(name):
    return name is None or any(s in name.lower() for s in SOFTWARE_GL)

render_env = os.environ.copy()
render_env["MUJOCO_GL"] = "egl"
renderer = gl_renderer(render_env)
if is_software(renderer):
    # mujoco.egl takes the first device that initialises, which may be Mesa's
    # even once the NVIDIA one is enumerated. Select a device explicitly.
    for device_id in range(4):
        candidate = dict(render_env, MUJOCO_EGL_DEVICE_ID=str(device_id))
        if not is_software(gl_renderer(candidate)):
            render_env = candidate
            renderer = gl_renderer(candidate)
            break
if is_software(renderer):
    raise RuntimeError(
        f"MuJoCo EGL resolved to {renderer!r}, not a GPU. Rendering would take "
        "hours. Confirm the driver's EGL library is installed -- "
        "`!ldconfig -p | grep libEGL_nvidia` should print a path; if it does "
        "not, `!apt-get install -y -qq libnvidia-gl-<version>` matching "
        "nvidia-smi, then rerun this cell.")
print("EGL renderer:", renderer,
      "(device", render_env.get("MUJOCO_EGL_DEVICE_ID", "auto") + ")")


In [ ]:
RENDERED_DIR = VLA_DIR / "data/robot_arm_learning_rendered"
RLDS_DIR = VLA_DIR / "data/robot_arm_learning"
run([PYTHON, CODE_DIR / "teleop/render_vla_dataset.py",
     "--input", CODE_DIR / "data",
     "--output", RENDERED_DIR,
     "--hz", str(SAMPLE_HZ), "--size", "256"], env=render_env)
run([PYTHON, CODE_DIR / "teleop/build_robot_arm_learning_rlds.py",
     "--rendered-dir", RENDERED_DIR,
     "--data-dir", RLDS_DIR,
     "--instruction", INSTRUCTION])


In [ ]:
# Sanity-check one rendered observation pair.
from IPython.display import display, Image
sample = sorted(RENDERED_DIR.glob("episode_*"))[0]
print(sample.name)
display(Image(filename=str(sample / "shoulder/00000.jpg"), width=320))
display(Image(filename=str(sample / "wrist/00000.jpg"), width=320))


## 5. Download the base model

In [ ]:
MODEL_DIR = VLA_DIR / "pretrained_models/prism-qwen25-extra-dinosiglip-224px-0_5b"
model_program = f'''
from huggingface_hub import snapshot_download
snapshot_download(repo_id={MODEL_REPO!r}, local_dir={str(MODEL_DIR)!r})
'''
run([PYTHON, "-c", model_program])


## 6. Fine-tune

The 16 GB default uses an effective batch size of 8 (`batch_size=2`, four accumulation steps). Gradient checkpointing and LoRA rank 64 are enabled. If CUDA reports an out-of-memory error, set `BATCH_SIZE=1` and `GRAD_ACCUM_STEPS=8`, then rerun this cell. W&B logging stays offline. On an Ampere-or-newer GPU, `uv pip install --python PYTHON flash_attn==2.5.5` is picked up automatically by the attention patch and speeds up training; it is optional. With `SAVE_TO_DRIVE=False`, run the export cell before the Colab runtime disconnects.

In [ ]:
from datetime import datetime, timezone

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path(DRIVE_OUTPUT)
else:
    OUTPUT_ROOT = VLA_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = "robot-arm-learning-colab-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

command = [
    VENV / "bin/torchrun", "--standalone", "--nnodes", "1", "--nproc-per-node", "1",
    VLA_DIR / "vla-scripts/finetune.py",
    "--vlm_path", MODEL_DIR,
    "--config_file_path", VLA_DIR / "pretrained_models/configs",
    "--data_root_dir", RLDS_DIR,
    "--dataset_name", "robot_arm_learning_panthera",
    "--run_root_dir", OUTPUT_ROOT,
    "--run_id_override", RUN_ID,
    "--use_film", "False",
    "--num_images_in_input", "2",
    "--use_proprio", "True",
    "--use_lora", "True",
    "--use_fz", "False",
    "--use_minivlm", "True",
    "--image_aug", "True",
    "--shuffle_buffer_size", "12000",
    "--num_steps_before_decay", str(max(1, int(MAX_STEPS * 0.8))),
    "--max_steps", str(MAX_STEPS),
    "--save_freq", str(SAVE_FREQ),
    "--save_latest_checkpoint_only", "True",
    "--merge_lora_during_training", "False",
    "--batch_size", str(BATCH_SIZE),
    "--grad_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--learning_rate", "2e-4",
    "--lora_rank", "64",
    "--use_pro_version", "True",
    "--use_gradient_checkpointing", "True",
    "--wandb_entity", "local",
    "--wandb_project", "robot-arm-learning-panthera",
]
train_env = os.environ.copy()
train_env.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "WANDB_MODE": "offline",
    "PYTHONPATH": str(VLA_DIR),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})
run(command, cwd=VLA_DIR, env=train_env)


## 7. Package the trained adapter

This archive contains the LoRA adapter, action head, proprio projector, processor files, and action-normalization statistics. The base model remains referenced separately.

In [ ]:
import tarfile

RUN_DIR = OUTPUT_ROOT / RUN_ID
if not (RUN_DIR / "lora_adapter").exists():
    raise RuntimeError("No saved adapter found. Ensure training reached SAVE_FREQ steps.")
ARCHIVE = ROOT / f"{RUN_ID}.tar.gz"
with tarfile.open(ARCHIVE, "w:gz") as archive:
    archive.add(RUN_DIR, arcname=RUN_ID)
print(f"Created {ARCHIVE} ({ARCHIVE.stat().st_size / 2**20:.1f} MiB)")

if SAVE_TO_DRIVE:
    destination = Path(DRIVE_OUTPUT) / ARCHIVE.name
    shutil.copy2(ARCHIVE, destination)
    print("Copied to", destination)
else:
    from google.colab import files
    files.download(str(ARCHIVE))
